# Teste Técnico — Engenharia de Dados
## Marketplace B2B (distribuidoras de alimentos e supermercados)

O notebook está organizado em três etapas: exploração dos dados, limpeza e
resolução dos quatro desafios de SQL.

A exploração e a limpeza precedem as análises porque parte das inconsistências
descritas no enunciado tem origem nos próprios dados (registros duplicados).
Calcular métricas sobre dados não tratados propagaria esses erros.

Observações técnicas:
- As consultas são executadas com `pandasql`, que utiliza SQLite. As datas são
  manipuladas com `strftime` e as razões multiplicadas por `1.0` para evitar a
  divisão inteira do SQLite.
- As janelas de tempo são definidas a partir da data máxima presente nos dados
  (novembro de 2024), e não da data de execução.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_style("whitegrid")

_TABELAS = ["orders", "order_items", "products", "sellers", "buyers", "payments"]

def pysqldf(query):
    """Executa uma consulta SQL sobre os DataFrames em memória, via SQLite.
    Reflete o estado atual das variáveis, inclusive após a etapa de limpeza."""
    conexao = sqlite3.connect(":memory:")
    try:
        ambiente = globals()
        for nome in _TABELAS:
            if isinstance(ambiente.get(nome), pd.DataFrame):
                ambiente[nome].to_sql(nome, conexao, index=False, if_exists="replace")
        return pd.read_sql_query(query, conexao)
    finally:
        conexao.close()

In [ ]:
BASE = "/content"   # no Colab; para execução local, ajuste para o diretório dos CSVs

orders      = pd.read_csv(f"{BASE}/orders.csv")
order_items = pd.read_csv(f"{BASE}/order_items.csv")
products    = pd.read_csv(f"{BASE}/products.csv")
sellers     = pd.read_csv(f"{BASE}/sellers.csv")
buyers      = pd.read_csv(f"{BASE}/buyers.csv")
payments    = pd.read_csv(f"{BASE}/payments.csv")

# 1. Exploração dos dados

A exploração segue duas frentes: verificação de integridade (tipos, chaves,
referências e consistência entre tabelas) e caracterização do negócio
(distribuições e tendências).

## 1.1 Estrutura das tabelas
Dimensões, tipos de dados e amostra de cada tabela.

In [ ]:
for nome, df in [("orders",orders),("order_items",order_items),
                 ("products",products),("sellers",sellers),
                 ("buyers",buyers),("payments",payments)]:
    print(f"\n===== {nome}  ({df.shape[0]:,} linhas x {df.shape[1]} colunas) =====")
    print(df.dtypes)
    display(df.head(3))

## 1.2 Valores ausentes
Contagem de valores nulos por tabela.

In [ ]:
for nome, df in [("orders",orders),("order_items",order_items),("products",products),
                 ("sellers",sellers),("buyers",buyers),("payments",payments)]:
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    print(f"{nome:12s}: {'sem nulos' if nulos.empty else dict(nulos)}")

O único campo com valores ausentes é `payments.paid_at`. A consulta abaixo
mostra que a ausência ocorre apenas em pagamentos não concluídos (`refunded`,
`failed`, `pending`) e nunca em pagamentos com status `paid`. Trata-se de
ausência esperada, e não de dado faltante.

In [ ]:
pysqldf("""
    SELECT status,
           COUNT(*)              AS qtd,
           SUM(paid_at IS NULL)  AS sem_data_pagamento
    FROM payments
    GROUP BY status
    ORDER BY qtd DESC
""")

## 1.3 Distribuição de status dos pedidos
A distribuição define quais status representam receita efetiva (`completed` e
`delivered`).

In [ ]:
st = pysqldf("SELECT status, COUNT(*) AS qtd FROM orders GROUP BY status ORDER BY qtd DESC")

plt.figure(figsize=(8,4))
cores = ['#2a9d8f' if s in ('completed','delivered') else '#e76f51' for s in st['status']]
plt.bar(st['status'], st['qtd'], color=cores)
for i, v in enumerate(st['qtd']):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom', fontsize=9)
plt.title("Distribuição de status dos pedidos")
plt.ylabel("nº de pedidos"); plt.tight_layout(); plt.show()

taxa = st.loc[st.status.isin(['cancelled','refunded']),'qtd'].sum()/st['qtd'].sum()
print(f"Cancelamentos e estornos: {taxa:.1%} dos pedidos")

## 1.4 Unicidade das chaves primárias
Verificação de duplicidade na coluna `id` de cada tabela.

In [ ]:
for t in ["orders","order_items","products","sellers","buyers","payments"]:
    dup = pysqldf(f"SELECT COUNT(*) c FROM (SELECT id FROM {t} GROUP BY id HAVING COUNT(*)>1)").iloc[0,0]
    print(f"{t:12s}: {dup} id(s) duplicado(s)")

Há duplicidade em `orders` e `buyers`. Os registros duplicados apresentam
valores divergentes entre si:

In [ ]:
display(pysqldf("SELECT * FROM orders WHERE id IN (SELECT id FROM orders GROUP BY id HAVING COUNT(*)>1) ORDER BY id"))
display(pysqldf("SELECT * FROM buyers WHERE id IN (SELECT id FROM buyers GROUP BY id HAVING COUNT(*)>1) ORDER BY id"))

Em `orders`, os pares divergem em `total_value` e `created_at`; em `buyers`, há
variação no nome (`Brasil` e `Brasilis`). O padrão é compatível com registros
ingeridos mais de uma vez a partir da origem, com pequenas diferenças. O
tratamento é feito na etapa de limpeza.

## 1.5 Integridade referencial
Verificação de chaves estrangeiras sem correspondência.

In [ ]:
checks = {
    "order_items -> orders":   "SELECT COUNT(*) FROM order_items oi LEFT JOIN orders o ON o.id=oi.order_id WHERE o.id IS NULL",
    "order_items -> products": "SELECT COUNT(*) FROM order_items oi LEFT JOIN products p ON p.id=oi.product_id WHERE p.id IS NULL",
    "orders -> sellers":       "SELECT COUNT(*) FROM orders o LEFT JOIN sellers s ON s.id=o.seller_id WHERE s.id IS NULL",
    "orders -> buyers":        "SELECT COUNT(*) FROM orders o LEFT JOIN buyers b ON b.id=o.buyer_id WHERE b.id IS NULL",
}
for desc, sql in checks.items():
    print(f"{desc:24s}: {pysqldf(sql).iloc[0,0]} registro(s) sem correspondência")

Não há registros órfãos. As inconsistências restringem-se à duplicidade de
chaves identificada em 1.4.

## 1.6 Consistência entre tabelas
Comparação dos valores de pedido registrados em `order_items`, `orders` e
`payments`.

In [ ]:
pysqldf("""
WITH itens AS (
    SELECT order_id,
           SUM(unit_price*qty)                        AS bruto,
           SUM(unit_price*qty - COALESCE(discount,0)) AS liquido
    FROM order_items GROUP BY order_id
),
pay AS (SELECT order_id, SUM(amount) AS pago FROM payments GROUP BY order_id)
SELECT
    SUM(CASE WHEN ABS(o.total_value - i.liquido) < 0.01 THEN 1 ELSE 0 END) AS tv_igual_liquido,
    SUM(CASE WHEN ABS(o.total_value - pay.pago)  < 0.01 THEN 1 ELSE 0 END) AS tv_igual_pagamento,
    COUNT(*) AS total
FROM orders o
JOIN itens i ON i.order_id = o.id
LEFT JOIN pay ON pay.order_id = o.id
""")

`orders.total_value` corresponde ao valor líquido (bruto menos desconto) e
coincide com o total dos pagamentos, exceto nos pedidos duplicados. Em razão
disso, o faturamento bruto é definido como `SUM(unit_price * qty)`.

## 1.7 Valores fora do domínio esperado
Preços e quantidades não positivos e descontos inconsistentes.

In [ ]:
pysqldf("""
SELECT
    SUM(unit_price <= 0)              AS preco_zero_ou_neg,
    SUM(qty <= 0)                     AS qtd_zero_ou_neg,
    SUM(discount < 0)                 AS desconto_negativo,
    SUM(discount > unit_price * qty)  AS desconto_maior_que_bruto
FROM order_items
""")

Nenhuma ocorrência.

## 1.8 Itens por pedido
Distribuição do número de itens por pedido. Relevante para o Desafio 4.

In [ ]:
ipo = pysqldf("""
    SELECT n_itens, COUNT(*) AS qtd
    FROM (SELECT order_id, COUNT(*) AS n_itens FROM order_items GROUP BY order_id)
    GROUP BY n_itens ORDER BY n_itens
""")
plt.figure(figsize=(8,4)); plt.bar(ipo['n_itens'], ipo['qtd'], color='#264653')
for i, r in ipo.iterrows(): plt.text(r['n_itens'], r['qtd'], f"{r['qtd']:,}", ha='center', va='bottom', fontsize=9)
plt.title("Itens por pedido"); plt.xlabel("nº de itens no pedido"); plt.ylabel("nº de pedidos")
plt.tight_layout(); plt.show()

um_item = ipo.loc[ipo.n_itens==1,'qtd'].iloc[0]
print(f"Pedidos com um único item: {um_item:,} ({um_item/ipo['qtd'].sum():.0%}). "
      f"Nesses pedidos, o item é, por definição, o de maior valor.")

## 1.9 Distribuição do desconto por pedido
Percentual de desconto sobre o valor bruto de cada pedido. A linha em 40%
corresponde ao limite adotado no Desafio 3.

In [ ]:
desc = pysqldf("""
WITH a AS (SELECT order_id, SUM(COALESCE(discount,0)) d, SUM(unit_price*qty) b FROM order_items GROUP BY order_id)
SELECT d*100.0/b AS pct FROM a WHERE b > 0
""")
plt.figure(figsize=(8,4)); plt.hist(desc['pct'], bins=40, color='#457b9d', edgecolor='white')
plt.axvline(40, color='#c1121f', ls='--', lw=2, label='limite de 40%')
plt.title("Percentual de desconto por pedido"); plt.xlabel("% de desconto"); plt.ylabel("nº de pedidos")
plt.legend(); plt.tight_layout(); plt.show()

print(f"Pedidos acima de 40% de desconto: {(desc['pct']>40).sum():,}. "
      f"Máximo observado: {desc['pct'].max():.1f}%")

## 1.10 Evolução do GMV mensal
GMV por mês, considerados os pedidos `completed` e `delivered`.

In [ ]:
gmv_mes = pysqldf("""
    SELECT strftime('%Y-%m', o.created_at) AS mes, SUM(oi.unit_price*oi.qty) AS gmv
    FROM orders o JOIN order_items oi ON oi.order_id = o.id
    WHERE o.status IN ('completed','delivered')
    GROUP BY mes ORDER BY mes
""")
plt.figure(figsize=(10,4))
plt.plot(gmv_mes['mes'], gmv_mes['gmv']/1e6, marker='o', color='#2a9d8f')
plt.scatter([gmv_mes['mes'].iloc[-1]], [gmv_mes['gmv'].iloc[-1]/1e6], color='#c1121f', s=90, zorder=5)
plt.annotate("novembro incompleto", (gmv_mes['mes'].iloc[-1], gmv_mes['gmv'].iloc[-1]/1e6),
             textcoords="offset points", xytext=(-55,10), color='#c1121f')
plt.title("GMV mensal (R$ milhões)"); plt.ylabel("GMV (R$ mi)")
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

O GMV mensal é estável, em torno de R$ 65 milhões. A redução no último ponto
decorre de novembro de 2024 estar incompleto (dados até o dia 29). A mesma
limitação afeta o trimestre corrente no Desafio 2.

## 1.11 Concentração de GMV por seller
GMV acumulado por seller, ordenado do maior para o menor.

In [ ]:
g = pysqldf("""
    SELECT o.seller_id, SUM(oi.unit_price*oi.qty) AS gmv
    FROM orders o JOIN order_items oi ON oi.order_id = o.id
    WHERE o.status IN ('completed','delivered')
    GROUP BY o.seller_id ORDER BY gmv DESC
""")
g['pct_gmv_acum'] = g['gmv'].cumsum() / g['gmv'].sum() * 100
g['pct_sellers']  = (np.arange(len(g)) + 1) / len(g) * 100

plt.figure(figsize=(7,5))
plt.plot(g['pct_sellers'], g['pct_gmv_acum'], color='#e76f51', lw=2)
plt.plot([0,100],[0,100], ls=':', color='gray', label='distribuição uniforme')
plt.title("GMV acumulado por seller"); plt.xlabel("% de sellers (do maior ao menor)")
plt.ylabel("% do GMV acumulado"); plt.legend(); plt.tight_layout(); plt.show()

top20 = g.loc[g['pct_sellers']<=20, 'pct_gmv_acum'].max()
print(f"Os 20% maiores sellers concentram {top20:.0f}% do GMV.")

Os 20% maiores sellers concentram cerca de 43% do GMV, concentração inferior
à proporção de 80/20 comumente assumida.

## 1.12 Correlação entre variáveis numéricas
Correlação entre quantidade, preço unitário, desconto e valor de linha em
`order_items`.

In [ ]:
oi = order_items.copy()
oi['valor_linha'] = oi['unit_price'] * oi['qty']
corr = oi[['qty','unit_price','discount','valor_linha']].corr()

plt.figure(figsize=(5.5,4.5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', linewidths=.5)
plt.title("Correlação — order_items"); plt.tight_layout(); plt.show()

As correlações são coerentes com a definição das variáveis: `valor_linha`
varia com `qty` e `unit_price`, e `discount` acompanha o valor de linha. Não se
observa relação inesperada entre os campos.

## 1.13 Síntese
Integridade: referências consistentes, ausência de valores fora do domínio e
valores nulos restritos a um caso esperado. O ponto a corrigir é a duplicidade
de registros em `orders` e `buyers`, tratada na próxima etapa.

Negócio: cancelamentos e estornos em torno de 15%, GMV mensal estável e
concentração de sellers moderada.

# 2. Limpeza dos dados

Os identificadores duplicados em `orders` multiplicam linhas na junção com
`order_items`: cada item passa a ser contado uma vez por linha duplicada do
pedido. A consulta abaixo demonstra o efeito.

In [ ]:
pysqldf("""
    SELECT oi.order_id, COUNT(*) AS linhas_no_join
    FROM orders o JOIN order_items oi ON oi.order_id = o.id
    WHERE o.id IN (79990, 79991)
    GROUP BY oi.order_id
""")
# Cada item é contado duas vezes em razão da duplicidade do id do pedido em orders.

Critério de deduplicação: manter um registro por `id`, preservando o mais
recente por `created_at`. Em um cenário real, a regra de reconciliação seria
confirmada com a origem do dado.

In [ ]:
antes_orders, antes_buyers = len(orders), len(buyers)

orders = (orders.sort_values("created_at")
                .drop_duplicates(subset="id", keep="last")
                .reset_index(drop=True))
buyers = (buyers.sort_values("created_at")
                .drop_duplicates(subset="id", keep="last")
                .reset_index(drop=True))

print(f"orders: {antes_orders:,} -> {len(orders):,}  ({antes_orders-len(orders)} removido[s])")
print(f"buyers: {antes_buyers:,} -> {len(buyers):,}  ({antes_buyers-len(buyers)} removido[s])")

Após a deduplicação, a multiplicação de linhas deixa de ocorrer. Como
`pysqldf` lê as variáveis em memória, as etapas seguintes utilizam os dados já
tratados.

In [ ]:
pysqldf("""
    SELECT oi.order_id, COUNT(*) AS linhas_no_join
    FROM orders o JOIN order_items oi ON oi.order_id = o.id
    WHERE o.id IN (79990, 79991)
    GROUP BY oi.order_id
""")

# 3. Colunas e premissas

| Tabela | Colunas relevantes |
|--------|--------------------|
| `orders` | `id`, `seller_id`, `buyer_id`, `status`, `created_at`, `total_value` (líquido) |
| `order_items` | `id`, `order_id`, `product_id`, `qty`, `unit_price`, `discount` (total da linha) |
| `products` | `id`, `name`, `category`, `seller_id`, `active`, `unit_cost` |
| `sellers` | `id`, `name`, `state`, `plan`, `created_at` |
| `buyers` | `id`, `name`, `city`, `state`, `segment`, `created_at` |
| `payments` | `id`, `order_id`, `paid_at`, `amount`, `method`, `status` |

Premissas: faturamento e GMV brutos correspondem a `SUM(unit_price * qty)`; o
seller é atribuído no nível do pedido (`orders.seller_id`); as janelas de tempo
são referenciadas em `MAX(created_at)`.

# Desafio 1 — Faturamento mensal dos últimos 12 meses

Faturamento bruto por mês, considerando apenas pedidos `completed` ou
`delivered`, com a quantidade de pedidos e o ticket médio, ordenado do mês mais
recente ao mais antigo.

A agregação ocorre primeiro no nível de pedido, de modo que a contagem
represente pedidos e não itens. A janela de doze meses é definida a partir da
data máxima. A coluna `faturamento_liquido` é incluída como referência
complementar.

In [ ]:
query_desafio1 = """
WITH pedidos_validos AS (
    SELECT o.id AS order_id, o.created_at,
           SUM(oi.unit_price * oi.qty)                           AS valor_bruto,
           SUM(oi.unit_price * oi.qty - COALESCE(oi.discount,0)) AS valor_liquido
    FROM orders AS o
    JOIN order_items AS oi ON oi.order_id = o.id
    WHERE o.created_at IS NOT NULL
      AND lower(trim(o.status)) IN ('completed', 'delivered')
    GROUP BY o.id, o.created_at
),
referencia AS (SELECT MAX(created_at) AS data_max FROM pedidos_validos)
SELECT
    strftime('%Y-%m', p.created_at)                AS mes,
    COUNT(*)                                       AS qtd_pedidos,
    ROUND(SUM(p.valor_bruto), 2)                   AS faturamento_bruto,
    ROUND(SUM(p.valor_liquido), 2)                 AS faturamento_liquido,
    ROUND(SUM(p.valor_bruto) * 1.0 / COUNT(*), 2)  AS ticket_medio
FROM pedidos_validos AS p
CROSS JOIN referencia AS r
WHERE strftime('%Y-%m', p.created_at) >= strftime('%Y-%m', date(r.data_max, '-11 months'))
GROUP BY mes
ORDER BY mes DESC;
"""
pysqldf(query_desafio1)

# Desafio 2 — Sellers com maior crescimento de GMV

Os dez sellers com maior crescimento de GMV entre o trimestre corrente e o
anterior, restritos aos que registraram ao menos 50 pedidos em ambos os
trimestres.

O trimestre é identificado por um índice contínuo (`ano*4 + (mês-1)/3`), que
ordena corretamente a passagem de ano. Os valores dos dois trimestres são
dispostos em colunas e comparados. Pedidos `cancelled` e `refunded` são
excluídos; a condição `gmv_anterior > 0` evita divisão por zero.

In [ ]:
query_desafio2 = """
WITH pedidos AS (
    SELECT o.seller_id, o.id AS order_id,
           CAST(strftime('%Y', o.created_at) AS INTEGER) * 4
               + (CAST(strftime('%m', o.created_at) AS INTEGER) - 1) / 3 AS idx_tri,
           SUM(oi.unit_price * oi.qty) AS valor_pedido
    FROM orders AS o
    JOIN order_items AS oi ON oi.order_id = o.id
    WHERE o.created_at IS NOT NULL
      AND lower(trim(o.status)) NOT IN ('cancelled', 'refunded')
    GROUP BY o.id, o.seller_id, idx_tri
),
ref AS (SELECT MAX(idx_tri) AS idx_atual FROM pedidos),
por_periodo AS (
    SELECT p.seller_id,
           CASE WHEN p.idx_tri = r.idx_atual THEN 'atual' ELSE 'anterior' END AS periodo,
           COUNT(*)            AS qtd_pedidos,
           SUM(p.valor_pedido) AS gmv
    FROM pedidos AS p CROSS JOIN ref AS r
    WHERE p.idx_tri IN (r.idx_atual, r.idx_atual - 1)
    GROUP BY p.seller_id, periodo
),
pivot AS (
    SELECT seller_id,
        SUM(CASE WHEN periodo='anterior' THEN gmv         END) AS gmv_ant,
        SUM(CASE WHEN periodo='anterior' THEN qtd_pedidos END) AS ped_ant,
        SUM(CASE WHEN periodo='atual'    THEN gmv         END) AS gmv_atu,
        SUM(CASE WHEN periodo='atual'    THEN qtd_pedidos END) AS ped_atu
    FROM por_periodo GROUP BY seller_id
)
SELECT
    s.name  AS seller,
    s.state AS estado,
    ROUND(pv.gmv_ant, 2) AS gmv_trim_anterior,
    ROUND(pv.gmv_atu, 2) AS gmv_trim_atual,
    ROUND((pv.gmv_atu - pv.gmv_ant) * 100.0 / pv.gmv_ant, 2) AS crescimento_pct
FROM pivot AS pv
JOIN sellers AS s ON s.id = pv.seller_id
WHERE COALESCE(pv.ped_ant,0) >= 50
  AND COALESCE(pv.ped_atu,0) >= 50
  AND pv.gmv_ant > 0
ORDER BY crescimento_pct DESC
LIMIT 10;
"""
pysqldf(query_desafio2)

## Trimestre corrente incompleto
Os dados terminam em 29 de novembro de 2024. O trimestre corrente (4º de 2024)
cobre 60 dias observados, contra 92 do trimestre anterior. A comparação direta
subestima o período corrente, o que explica a predominância de variações
negativas.

A consulta a seguir normaliza o GMV pelo número de dias observados em cada
trimestre. Sob essa métrica, os mesmos sellers apresentam crescimento. A
ordenação se mantém, pois o fator de correção é comum a todos os sellers.

In [ ]:
query_desafio2_normalizado = """
WITH pedidos AS (
    SELECT o.seller_id, o.id AS order_id, date(o.created_at) AS dia,
           CAST(strftime('%Y', o.created_at) AS INTEGER) * 4
               + (CAST(strftime('%m', o.created_at) AS INTEGER) - 1) / 3 AS idx_tri,
           SUM(oi.unit_price * oi.qty) AS valor_pedido
    FROM orders AS o
    JOIN order_items AS oi ON oi.order_id = o.id
    WHERE o.created_at IS NOT NULL
      AND lower(trim(o.status)) NOT IN ('cancelled', 'refunded')
    GROUP BY o.id, o.seller_id, idx_tri, dia
),
ref AS (SELECT MAX(idx_tri) AS idx_atual FROM pedidos),
dias_tri AS (SELECT idx_tri, COUNT(DISTINCT dia) AS n_dias FROM pedidos GROUP BY idx_tri),
por_periodo AS (
    SELECT p.seller_id,
           CASE WHEN p.idx_tri = r.idx_atual THEN 'atual' ELSE 'anterior' END AS periodo,
           COUNT(*) AS qtd_pedidos, SUM(p.valor_pedido) AS gmv, MAX(d.n_dias) AS n_dias
    FROM pedidos AS p
    CROSS JOIN ref AS r
    JOIN dias_tri AS d ON d.idx_tri = p.idx_tri
    WHERE p.idx_tri IN (r.idx_atual, r.idx_atual - 1)
    GROUP BY p.seller_id, periodo
),
pivot AS (
    SELECT seller_id,
        SUM(CASE WHEN periodo='anterior' THEN gmv / n_dias END) AS gmv_dia_ant,
        SUM(CASE WHEN periodo='anterior' THEN qtd_pedidos  END) AS ped_ant,
        SUM(CASE WHEN periodo='atual'    THEN gmv / n_dias END) AS gmv_dia_atu,
        SUM(CASE WHEN periodo='atual'    THEN qtd_pedidos  END) AS ped_atu
    FROM por_periodo GROUP BY seller_id
)
SELECT
    s.name  AS seller,
    s.state AS estado,
    ROUND(pv.gmv_dia_ant, 2) AS gmv_diario_anterior,
    ROUND(pv.gmv_dia_atu, 2) AS gmv_diario_atual,
    ROUND((pv.gmv_dia_atu - pv.gmv_dia_ant) * 100.0 / pv.gmv_dia_ant, 2) AS crescimento_pct
FROM pivot AS pv
JOIN sellers AS s ON s.id = pv.seller_id
WHERE COALESCE(pv.ped_ant,0) >= 50 AND COALESCE(pv.ped_atu,0) >= 50 AND pv.gmv_dia_ant > 0
ORDER BY crescimento_pct DESC
LIMIT 10;
"""
pysqldf(query_desafio2_normalizado)

O gráfico compara, para os mesmos sellers, o crescimento obtido pela comparação
direta entre trimestres e pela versão normalizada por dia observado.

In [ ]:
cresc_direto = pysqldf(query_desafio2)[["seller", "crescimento_pct"]].rename(
    columns={"crescimento_pct": "direto"})
cresc_norm = pysqldf(query_desafio2_normalizado)[["seller", "crescimento_pct"]].rename(
    columns={"crescimento_pct": "normalizado"})
comparacao = cresc_direto.merge(cresc_norm, on="seller")

x = np.arange(len(comparacao)); largura = 0.4
plt.figure(figsize=(11,5))
plt.bar(x - largura/2, comparacao["direto"], largura, label="comparação direta", color="#adb5bd")
plt.bar(x + largura/2, comparacao["normalizado"], largura, label="normalizado por dia", color="#2a9d8f")
plt.axhline(0, color="black", lw=0.8)
plt.xticks(x, comparacao["seller"], rotation=45, ha="right")
plt.ylabel("crescimento de GMV (%)")
plt.title("Crescimento trimestral por seller: comparação direta vs. normalizada por dia")
plt.legend(); plt.tight_layout(); plt.show()

# Desafio 3 — Pedidos com desconto acima de 40% do valor bruto

Pedidos em que a soma dos descontos dos itens excede 40% do valor bruto, com o
seller responsável e a data. Pedidos cancelados são excluídos.

O desconto e o valor bruto são agregados por pedido. A condição é escrita como
`desconto_total > 0.40 * valor_bruto`, o que dispensa divisão na comparação;
`valor_bruto > 0` protege o cálculo do percentual exibido.

In [ ]:
query_desafio3 = """
WITH agg_pedido AS (
    SELECT oi.order_id,
           SUM(COALESCE(oi.discount, 0)) AS desconto_total,
           SUM(oi.unit_price * oi.qty)   AS valor_bruto
    FROM order_items AS oi
    GROUP BY oi.order_id
)
SELECT
    o.id AS order_id, s.name AS seller, o.created_at AS data_pedido,
    ROUND(a.valor_bruto, 2)                            AS valor_bruto,
    ROUND(a.desconto_total, 2)                         AS desconto_total,
    ROUND(a.desconto_total * 100.0 / a.valor_bruto, 2) AS pct_desconto
FROM agg_pedido AS a
JOIN orders  AS o ON o.id = a.order_id
JOIN sellers AS s ON s.id = o.seller_id
WHERE lower(trim(o.status)) <> 'cancelled'
  AND a.valor_bruto > 0
  AND a.desconto_total > 0.40 * a.valor_bruto
ORDER BY pct_desconto DESC;
"""
resultado = pysqldf(query_desafio3)
print("Pedidos sinalizados:", len(resultado))
resultado.head(15)

A distribuição dos pedidos sinalizados não é uniforme entre os sellers. O
gráfico e o ranking abaixo apresentam os sellers com maior número de pedidos
nessa condição.

In [ ]:
ranking_sellers = pysqldf("""
WITH agg AS (SELECT order_id, SUM(COALESCE(discount,0)) AS d, SUM(unit_price*qty) AS b
             FROM order_items GROUP BY order_id)
SELECT s.name AS seller, s.state AS estado, COUNT(*) AS pedidos_sinalizados
FROM agg a
JOIN orders  o ON o.id = a.order_id
JOIN sellers s ON s.id = o.seller_id
WHERE lower(trim(o.status)) <> 'cancelled' AND a.b > 0 AND a.d > 0.40 * a.b
GROUP BY s.name, s.state
ORDER BY pedidos_sinalizados DESC
LIMIT 10;
""")

plt.figure(figsize=(9,4.5))
posicoes = range(len(ranking_sellers))
plt.barh(list(posicoes), ranking_sellers["pedidos_sinalizados"], color="#e76f51")
plt.yticks(list(posicoes), ranking_sellers["seller"]); plt.gca().invert_yaxis()
for i, v in enumerate(ranking_sellers["pedidos_sinalizados"]):
    plt.text(v, i, f" {v}", va="center", fontsize=9)
plt.title("Sellers com mais pedidos de desconto acima de 40%")
plt.xlabel("nº de pedidos sinalizados"); plt.tight_layout(); plt.show()

ranking_sellers

# Desafio 4 — Produtos de alto volume que nunca são o item de maior valor

Produtos com mais de 1.000 unidades vendidas que nunca figuram como item de
maior valor unitário em nenhum pedido.

Para cada pedido, `MAX(unit_price) OVER (PARTITION BY order_id)` determina o maior
preço unitário. Um item é o de maior valor quando `unit_price` iguala esse
máximo; a igualdade trata empates, incluindo todos os itens que atingem o
máximo.

In [ ]:
query_desafio4 = """
WITH item_com_max AS (
    SELECT oi.product_id, oi.qty, oi.unit_price,
           MAX(oi.unit_price) OVER (PARTITION BY oi.order_id) AS max_preco_pedido
    FROM order_items AS oi
),
por_produto AS (
    SELECT product_id,
           SUM(qty) AS unidades_vendidas,
           SUM(CASE WHEN unit_price = max_preco_pedido THEN 1 ELSE 0 END) AS vezes_no_topo
    FROM item_com_max
    GROUP BY product_id
)
SELECT pr.id AS product_id, pr.name AS produto, p.unidades_vendidas
FROM por_produto AS p
JOIN products AS pr ON pr.id = p.product_id
WHERE p.unidades_vendidas > 1000
  AND p.vezes_no_topo = 0
ORDER BY unidades_vendidas DESC;
"""
resultado = pysqldf(query_desafio4)
print("Produtos encontrados:", len(resultado))
resultado

## Avaliação do resultado
A consulta não retorna registros. Como o enunciado emprega expressões distintas
("item mais vendido", "item de maior valor", "maior valor unitário"), a mesma
lógica é aplicada a quatro definições de item mais importante do pedido: preço
unitário, valor de linha (preço × quantidade), valor líquido e quantidade.

In [ ]:
definicoes = {
    "preço unitário": "unit_price",
    "valor de linha": "unit_price*qty",
    "valor líquido":  "unit_price*qty - COALESCE(discount,0)",
    "quantidade":     "qty",
}
linhas = []
for rotulo, metrica in definicoes.items():
    r = pysqldf(f"""
        WITH im AS (
            SELECT product_id, qty, ({metrica}) AS m,
                   MAX({metrica}) OVER (PARTITION BY order_id) AS mx
            FROM order_items
        ),
        pp AS (
            SELECT product_id, SUM(qty) AS unidades, COUNT(*) AS aparicoes,
                   SUM(CASE WHEN m = mx THEN 1 ELSE 0 END) AS vezes_topo
            FROM im GROUP BY product_id
        )
        SELECT SUM(CASE WHEN unidades>1000 AND vezes_topo=0 THEN 1 ELSE 0 END) AS nunca_topo,
               ROUND(MIN(CASE WHEN unidades>1000 THEN vezes_topo*100.0/aparicoes END),1) AS menor_taxa_pct
        FROM pp
    """)
    linhas.append({"definicao_de_topo": rotulo,
                   "produtos_nunca_no_topo": int(r["nunca_topo"][0]),
                   "menor_taxa_no_topo_pct": r["menor_taxa_pct"][0]})
pd.DataFrame(linhas)

O histograma abaixo mostra a distribuição da taxa com que cada produto de alto
volume é o item de maior preço do seu pedido. A condição do desafio exigiria
produtos em 0%.

In [ ]:
taxa = pysqldf("""
    WITH im AS (
        SELECT product_id, unit_price,
               MAX(unit_price) OVER (PARTITION BY order_id) AS mx
        FROM order_items
    ),
    pp AS (
        SELECT product_id, COUNT(*) AS aparicoes,
               SUM(CASE WHEN unit_price = mx THEN 1 ELSE 0 END) AS vezes_topo
        FROM im GROUP BY product_id
    ),
    un AS (SELECT product_id, SUM(qty) AS unidades FROM order_items GROUP BY product_id)
    SELECT pp.vezes_topo * 100.0 / pp.aparicoes AS taxa_topo
    FROM pp JOIN un ON un.product_id = pp.product_id
    WHERE un.unidades > 1000
""")
plt.figure(figsize=(8,4))
plt.hist(taxa["taxa_topo"], bins=30, color="#264653", edgecolor="white")
plt.axvline(0, color="#c1121f", ls="--", lw=2, label="condição do desafio (0%)")
plt.title("Frequência com que cada produto é o item de maior preço do pedido")
plt.xlabel("% de pedidos em que o produto é o item de maior preço")
plt.ylabel("nº de produtos"); plt.legend(); plt.tight_layout(); plt.show()

Sob todas as quatro definições, nenhum produto com mais de 1.000 unidades deixa
de ser, em algum pedido, o item mais importante; a menor taxa observada fica em
torno de 27%, e o histograma não apresenta produtos próximos de 0%.

A causa é estrutural. A taxa média observada de um produto ser o item de maior
preço (37%) coincide com a taxa esperada caso esse item fosse sorteado ao acaso
dentro de cada pedido — o inverso do número de itens do pedido, cuja média também
é 37%. Os preços, portanto, distribuem-se de forma efetivamente aleatória entre os
produtos, sem que nenhum seja sistematicamente o mais barato. Além disso, cerca de
20% dos pedidos contêm um único item, no qual o produto é necessariamente o de
maior valor.

Com os dados fornecidos, nenhum produto atende à condição, sob nenhuma das
interpretações. O enunciado pressupõe a existência desse produto, o que sugere que
a análise foi concebida sobre um conjunto de dados com outra distribuição de
preços. A resposta é o conjunto vazio, acompanhada da justificativa acima.

# Notas finais
- Sequência adotada: exploração, limpeza e análise sobre dados tratados.
- Janelas de tempo referenciadas na data máxima dos dados; razões com fator
  `1.0`/`100.0` e proteção contra divisão por zero; normalização de status com
  `lower`/`trim`; `COALESCE` nos descontos.
- As consultas foram executadas sobre os dados fornecidos.